In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')


In [2]:
#load the data 
all = '/Users/sivhoung/Fraud_detection/paysim_dataset.csv'
his = "/Users/sivhoung/Fraud_detection/historical_data/part-00000-afbc1c86-0307-4cb4-96a5-8db1604ed007-c000.csv"
live='/Users/sivhoung/Fraud_detection/stream_simulation_data/part-00000-955a79b5-98bb-4146-838c-ed7dd5a9c43b-c000.csv'

all_data = pd.read_csv(all)
his_data = pd.read_csv(his)
live_data= pd.read_csv(live)


In [3]:
# 1. 基础结构检查
print(f"Dataset shape: {all_data.shape}")
print(all_data.info())

# 2. 关键业务分布检查（这是为了支持答辩中的“高危类型过滤”逻辑）
# 只有确认了 TRANSFER 和 CASH_OUT 是主要欺诈源，你的后续处理才显得专业
print("\n交易类型欺诈分布：")
print(all_data.groupby('type')['isFraud'].value_counts(normalize=True))

# 3. 欺诈占比检查
fraud_rate = all_data['isFraud'].mean()
print(f"\n整体欺诈率: {fraud_rate:.4%}")

Dataset shape: (6362620, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB
None

交易类型欺诈分布：
type      isFraud
CASH_IN   0          1.000000
CASH_OUT  0          0.998160
          1          0.001840
DEBIT     0          1.000000
PAYMENT   0          1.000000
TRANSFER  0          0.992312
          1          0.007688
Name: proportion, dtype: float64

整体欺诈率: 0.1291%


In [5]:
# 检查训练集是否依然存在严重的类别不平衡
print("\n训练数据标签分布：")
print(his_data['isfraud'].value_counts())

# 检查特征是否存在缺失值或无穷大值
print("\n缺失值检查：")
print(his_data.isnull().sum())


训练数据标签分布：
isfraud
0    5083466
1       6599
Name: count, dtype: int64

缺失值检查：
type                     0
amount                   0
nameorig                 0
oldbalanceorg            0
newbalanceorig           0
namedest                 0
oldbalancedest           0
newbalancedest           0
isfraud                  0
isflaggedfraud           0
transaction_timestamp    0
dtype: int64


In [6]:
# 检查是否有时间戳字段，这是实时流模拟的关键
print("\n实时数据示例（前5行）：")
print(live_data.head())

# 确认列名是否与 all_data 一致，防止后续推理引擎报错
print(f"\n列名校验: {list(live_data.columns) == list(all_data.columns)}")


实时数据示例（前5行）：
      type  amount     nameorig  oldbalanceorg  newbalanceorig     namedest  \
0  CASH_IN   14.40  C1108458753    11434608.13     11434622.53  C1587543488   
1  CASH_IN   17.33   C963690108     8964056.72      8964074.05   C245197553   
2  CASH_IN   21.57  C1410453091      104362.00       104383.57  C2142748198   
3  CASH_IN   37.11  C1843703160     1452790.24      1452827.35    C30079792   
4  CASH_IN   52.64  C1610950365     3468172.49      3468225.13   C778678237   

   oldbalancedest  newbalancedest  isfraud  isflaggedfraud  \
0        46093.00       124958.22        0               0   
1      1129421.51      1129404.18        0               0   
2      7251134.87      7289255.53        0               0   
3      7666143.68      7471854.78        0               0   
4         5255.00            0.00        0               0   

           transaction_timestamp  
0  2026-01-01T11:00:00.000+07:00  
1  2026-01-01T16:00:00.000+07:00  
2  2026-01-01T18:00:00.000+07:00 

In [7]:
print(f"Dataset shape: {all_data.shape}")
print(f"Batch Dataset: {his_data.shape}")
print(f"Live Dataset: {live_data.shape}")

Dataset shape: (6362620, 11)
Batch Dataset: (5090065, 11)
Live Dataset: (1272555, 11)
